In [ ]:
import pandas as pd
import numpy as np
import yaml
from stages.stage1_neglected_sector import NeglectedSectorScreener

print("==================================================")
print("🚀 [검증] Stage 1: 소외 섹터 발굴 로직 독립 테스트")
print("==================================================\n")

# 1. params.yaml 환경 모사
mock_params = {
    'stage1_return_weight': 0.5,
    'stage1_volume_weight': 0.5,
    'stage1_pass_ratio': 0.5  # 테스트를 위해 상위 50% 통과로 설정
}

# 2. loader가 제공할 '정제된 섹터 데이터' 모사 (Mock Data)
# 파이프라인은 이 데이터를 loader.get_sector_metrics() 등을 통해 받아왔다고 가정합니다.
mock_sector_data = pd.DataFrame({
    'sector': [
        'IT/소프트웨어',    # 하락세 진정, 거래대금 감소 (건전한 소외, 통과 기대)
        '바이오/헬스케어',  # 최근 1개월 낙폭 가속 (밸류트랩 경고 기대)
        '고배당/금융',      # 수익률 양호, 거래 활발 (소외되지 않음, 탈락 기대)
        '2차전지',          # 장기 하락, 거래대금 극감 (극심한 소외, 통과 기대)
        '건설/기계',        # 지속적인 구조적 하락 (밸류트랩 경고 기대)
        '필수소비재'        # 시장 평균 수준의 방어주
    ],
    'return_1m': [-0.01, -0.15, 0.04, -0.02, -0.08, 0.01], 
    'return_6m': [-0.20, -0.05, 0.10, -0.35, -0.10, 0.02], 
    'vol_prop_1m': [0.04, 0.09, 0.18, 0.05, 0.03, 0.10],   
    'vol_prop_1y': [0.15, 0.10, 0.12, 0.25, 0.05, 0.09]    
})

# 3. Stage 1 스크리너 실행
try:
    screener = NeglectedSectorScreener(params=mock_params)
    passed_sectors = screener.run(mock_sector_data)
    
    print("📊 [입력된 원본 섹터 데이터]")
    display(mock_sector_data)
    
    print("\n✅ [Stage 1 필터링 통과 결과 (상위 50%)]")
    display(passed_sectors)
    
    # 4. 밸류트랩 작동 확인
    value_traps = passed_sectors[passed_sectors['is_value_trap_warning'] == True]
    print("\n🔍 [밸류트랩(Value Trap) 방어선 검증]")
    if not value_traps.empty:
        print(f"⚠️ 경고: 다음 섹터는 낙폭이 가팔라지고 있어 주의가 필요합니다.\n -> {value_traps['sector'].tolist()}")
    else:
        print("✅ 통과된 섹터 중 급격한 낙폭 가속(Value Trap) 위험이 감지된 섹터는 없습니다.")

except Exception as e:
    print(f"❌ 에러 발생: {e}")

In [ ]:
import pandas as pd
import numpy as np
from datetime import date
from stages.stage2_sector_leaders import SectorLeaderScreener

print("==================================================")
print("🚀 [검증] Stage 2: 섹터 내 우량주 탐색 독립 테스트")
print("==================================================\n")

# 1. params.yaml 환경 모사
mock_params = {
    'roe_percentile_cutoff': 0.5,
    'roic_percentile_cutoff': 0.5,
    'op_margin_std_percentile_cutoff': 0.5,
    'op_margin_lookback_q': 8,
    'op_margin_min_quarters': 4
}

# 2. Loader 모사 (Mock Loader) - 외부 API 호출 없이 정해진 데이터 반환
class MockLoader:
    def get_ttm_financials(self, ticker: str, base_date: date) -> dict:
        data = {
            # A: 우량 성장주 (수익성 높음, 변동성 낮음) -> 통과 기대
            'TICKER_A': {'net_income': 200, 'total_equity': 1000, 'operating_income': 250, 'total_assets': 2000, 'total_liabilities': 1000},
            # B: 부실주 (수익성 낮음, 변동성 높음) -> 탈락 기대
            'TICKER_B': {'net_income': 50, 'total_equity': 1000, 'operating_income': 60, 'total_assets': 2000, 'total_liabilities': 1000},
            # C: 금융주 (자산/부채 개념이 달라 ROIC 계산 불가) -> 예외 구제 통과 기대
            'TICKER_C': {'net_income': 150, 'total_equity': 1000, 'operating_income': 180, 'total_assets': np.nan, 'total_liabilities': np.nan},
            # D: 신규 상장주 (데이터 기간 부족) -> 예외 구제 통과 기대
            'TICKER_D': {'net_income': 250, 'total_equity': 1000, 'operating_income': 300, 'total_assets': 2000, 'total_liabilities': 1000},
        }
        return data.get(ticker, {})

    def get_quarterly_op_margin_series(self, ticker: str, base_date: date, n_quarters: int = 8) -> list:
        data = {
            'TICKER_A': [0.10, 0.11, 0.10, 0.12, 0.11, 0.10, 0.11, 0.12], # 변동성 낮음
            'TICKER_B': [0.05, -0.02, 0.15, -0.10, 0.08, -0.05, 0.20, 0.01], # 변동성 높음
            'TICKER_C': [0.08, 0.09, 0.07, 0.08, 0.09, 0.08, 0.07, 0.08], # 변동성 낮음
            'TICKER_D': [0.15, 0.16, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan] # 데이터 부족(2분기)
        }
        return data.get(ticker, [np.nan] * n_quarters)

# 3. Stage 1을 통과했다고 가정할 종목 리스트 (Input Data)
mock_sector_tickers = pd.DataFrame({
    'ticker': ['TICKER_A', 'TICKER_B', 'TICKER_C', 'TICKER_D'],
    'sector': ['IT/소프트웨어', 'IT/소프트웨어', '고배당/금융', 'IT/소프트웨어']
})

# 4. Stage 2 스크리너 실행
try:
    screener = SectorLeaderScreener(params=mock_params)
    mock_loader = MockLoader()
    base_date = date(2026, 7, 31)
    
    passed_df = screener.run(mock_sector_tickers, mock_loader, base_date)
    
    print("✅ [Stage 2 필터링 통과 결과]")
    display(passed_df)

except Exception as e:
    print(f"❌ 에러 발생: {e}")

In [ ]:
import pandas as pd
import numpy as np
from datetime import date
from stages.stage3_fundamental_improve import FundamentalImproveScreener

print("==================================================")
print("🚀 [검증] Stage 3: 체질 개선(Turnaround) 독립 테스트")
print("==================================================\n")

# 1. params.yaml 환경 모사
mock_params = {
    'sga_lookback_quarters': 6,
    'require_sales_growth': True,
    'require_gpm_improvement': True,
    'require_inventory_turnover_up': True
}

# 2. Loader 모사 (Mock Loader) - 6개 분기(최근~5분기 전) 데이터 제공
class MockLoader:
    def get_quarterly_financials_series(self, ticker: str, base_date: date, n_quarters: int = 6) -> list:
        # 인덱스: 0(최근), 1(직전), ... 4(전년동기), 5(직전분기의 전년동기)
        
        # A: 진짜 턴어라운드 (매출 증가, 판관비율 감소, GPM 개선, 재고회전율 상승) -> 통과 기대
        # Q5/Q4 -> 매출 1000/1100, 판관비 200/220 (20%), 총이익 300/330 (30%), 재고 500/550 (회전율 2.0)
        # Q1/Q0 -> 매출 1200/1300, 판관비 180/195 (15%), 총이익 380/420 (32%), 재고 500/520 (회전율 2.5)
        data_a = [
            {'revenue': 1300, 'sga': 195, 'gross_profit': 420, 'inventory': 520}, # t=0 (최근)
            {'revenue': 1200, 'sga': 180, 'gross_profit': 380, 'inventory': 500}, # t=-1 (직전)
            {'revenue': 1150, 'sga': 200, 'gross_profit': 350, 'inventory': 520}, # t=-2
            {'revenue': 1100, 'sga': 220, 'gross_profit': 330, 'inventory': 550}, # t=-3
            {'revenue': 1100, 'sga': 220, 'gross_profit': 330, 'inventory': 550}, # t=-4 (전년동기)
            {'revenue': 1000, 'sga': 200, 'gross_profit': 300, 'inventory': 500}  # t=-5 (직전분기의 전년동기)
        ]
        
        # B: 불황형 흑자 (판관비율은 줄었으나 매출 감소, GPM 훼손) -> 탈락 기대
        data_b = [
            {'revenue': 800, 'sga': 120, 'gross_profit': 160, 'inventory': 600}, # t=0
            {'revenue': 900, 'sga': 135, 'gross_profit': 200, 'inventory': 600}, # t=-1
            {'revenue': 950, 'sga': 190, 'gross_profit': 250, 'inventory': 550}, # t=-2
            {'revenue': 1000, 'sga': 200, 'gross_profit': 300, 'inventory': 500},# t=-3
            {'revenue': 1000, 'sga': 200, 'gross_profit': 300, 'inventory': 500},# t=-4
            {'revenue': 1000, 'sga': 200, 'gross_profit': 300, 'inventory': 500} # t=-5
        ]

        # C: 금융주 (재고 및 매출원가/총이익 개념 없음) -> 예외 통과 기대
        data_c = [
            {'revenue': 2000, 'sga': 200, 'gross_profit': np.nan, 'inventory': np.nan}, # t=0
            {'revenue': 1900, 'sga': 250, 'gross_profit': np.nan, 'inventory': np.nan}, # t=-1
            {'revenue': 1800, 'sga': 250, 'gross_profit': np.nan, 'inventory': np.nan}, # t=-2
            {'revenue': 1800, 'sga': 250, 'gross_profit': np.nan, 'inventory': np.nan}, # t=-3
            {'revenue': 1800, 'sga': 250, 'gross_profit': np.nan, 'inventory': np.nan}, # t=-4
            {'revenue': 1800, 'sga': 300, 'gross_profit': np.nan, 'inventory': np.nan}  # t=-5
        ]

        # D: 데이터 부족
        data_d = [{'revenue': 1000, 'sga': 100}] * 3 # 3분기치만 존재

        if ticker == 'TICKER_A': return data_a
        elif ticker == 'TICKER_B': return data_b
        elif ticker == 'TICKER_C': return data_c
        elif ticker == 'TICKER_D': return data_d
        return []

# 3. Stage 2를 통과했다고 가정할 종목 리스트
mock_sector_tickers = pd.DataFrame({
    'ticker': ['TICKER_A', 'TICKER_B', 'TICKER_C', 'TICKER_D'],
    'sector': ['IT/소프트웨어', '제조업', '고배당/금융', '신규상장업']
})

# 4. Stage 3 스크리너 실행
try:
    screener = FundamentalImproveScreener(params=mock_params)
    mock_loader = MockLoader()
    base_date = date(2026, 7, 31)
    
    passed_df = screener.run(mock_sector_tickers, mock_loader, base_date)
    
    print("✅ [Stage 3 체질 개선 필터링 결과]")
    display(passed_df)

except Exception as e:
    print(f"❌ 에러 발생: {e}")

In [ ]:
import pandas as pd
import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta
from stages.stage4_valuation import ValuationScreener

print("==================================================")
print("🚀 [검증] Stage 4: 밸류에이션(Valuation) 독립 테스트")
print("==================================================\n")

# 1. params.yaml 환경 모사
mock_params = {
    'pbr_percentile_cutoff': 0.5,      # 섹터 내 PBR 하위 50% 통과
    'bps_growth_yoy_min': 0.0,         # BPS가 전년 대비 훼손되지 않을 것
    'roe_value_trap_threshold': 0.05   # ROE 5% 미만 + 저PBR = 밸류트랩
}

# 2. Loader 모사 (Mock Loader) - pykrx 기시산출값(펀더멘털 스냅샷) 제공
class MockLoader:
    def get_market_fundamental_cross_section(self, target_date: date) -> pd.DataFrame:
        if target_date.year == 2026: 
            # 현재 (t=0)
            data = {
                'ticker': ['TICKER_A', 'TICKER_B', 'TICKER_C', 'TICKER_D'],
                'bps': [11000, 9000, 12000, 1500],
                'pbr': [0.8, 0.5, 3.0, np.nan] # PBR 순위: B(1위) -> A(2위) -> C(3위), D는 제외
            }
        else: 
            # 1년 전 (t=-4)
            data = {
                'ticker': ['TICKER_A', 'TICKER_B', 'TICKER_C', 'TICKER_D'],
                'bps': [10000, 10000, 10000, 1000],
                'pbr': [0.9, 0.8, 2.5, np.nan]
            }
        return pd.DataFrame(data)

# 3. Stage 2, 3를 통과했다고 가정할 종목 리스트 (Input Data)
# 4단계는 ROE와 PBR을 페어링하므로, Stage 2에서 계산된 'roe'가 필수로 들어와야 함
mock_input_df = pd.DataFrame({
    'ticker': ['TICKER_A', 'TICKER_B', 'TICKER_C', 'TICKER_D'],
    'sector': ['제조업', '제조업', '제조업', '제조업'], # 동일 섹터로 묶어 상대 순위 확인
    'roe': [0.15, 0.03, 0.20, 0.10] 
})

# 4. Stage 4 스크리너 실행
try:
    screener = ValuationScreener(params=mock_params)
    mock_loader = MockLoader()
    base_date = date(2026, 7, 31)
    
    passed_df = screener.run(mock_input_df, mock_loader, base_date)
    
    print("📊 [Stage 4 입력 데이터 (ROE 포함)]")
    display(mock_input_df)
    
    print("\n✅ [Stage 4 밸류에이션 필터링 결과]")
    display(passed_df)

except Exception as e:
    print(f"❌ 에러 발생: {e}")

In [ ]:
import pandas as pd
import numpy as np
from datetime import date
from stages.stage5_financial_health import FinancialHealthScreener

print("==================================================")
print("🚀 [검증] Stage 5: 재무 건전성(Financial Health) 독립 테스트")
print("==================================================\n")

# 1. params.yaml 환경 모사
mock_params = {
    'debt_ratio_percentile_cutoff': 0.5,  # 섹터 내 부채비율 하위 50% 통과 (낮을수록 좋음)
    'interest_coverage_min': 1.5          # 영업이익이 이자비용의 1.5배 이상일 것
}

# 2. Loader 모사 (Mock Loader) - TTM 재무 데이터 반환
class MockLoader:
    def get_ttm_financials(self, ticker: str, base_date: date) -> dict:
        data = {
            # A (제조업): 부채비율 낮음(0.4), OCF > 순이익, 이자보상배율 10배 -> 통과 기대
            'TICKER_A': {'total_liabilities': 400, 'total_equity': 1000, 'operating_cash_flow': 150, 'net_income': 100, 'operating_income': 200, 'interest_expense': 20},
            
            # B (제조업): 부채비율 양호(0.5), OCF 흑자지만 순이익보다 작음(현금흐름 불량), 이자보상배율 양호 -> 탈락 기대
            'TICKER_B': {'total_liabilities': 500, 'total_equity': 1000, 'operating_cash_flow': 50, 'net_income': 200, 'operating_income': 250, 'interest_expense': 30},
            
            # C (금융업): 부채비율 매우 높음(10.0)이나 예외 처리됨, OCF > 순이익, 이자보상배율 양호 -> 예외 통과 기대
            'TICKER_C': {'total_liabilities': 10000, 'total_equity': 1000, 'operating_cash_flow': 500, 'net_income': 300, 'operating_income': 600, 'interest_expense': 100},
            
            # D (제조업): 부채비율 높음(2.0), OCF 양호, 이자보상배율 미달(0.8배) -> 탈락 기대
            'TICKER_D': {'total_liabilities': 2000, 'total_equity': 1000, 'operating_cash_flow': 100, 'net_income': 50, 'operating_income': 40, 'interest_expense': 50},
            
            # E (무차입 경영): 이자비용이 0 이하 -> 무한대 상환능력으로 통과 기대
            'TICKER_E': {'total_liabilities': 100, 'total_equity': 1000, 'operating_cash_flow': 200, 'net_income': 100, 'operating_income': 150, 'interest_expense': 0},
        }
        return data.get(ticker, {})

# 3. Stage 4를 통과했다고 가정할 종목 리스트 (Input Data)
mock_input_df = pd.DataFrame({
    'ticker': ['TICKER_A', 'TICKER_B', 'TICKER_C', 'TICKER_D', 'TICKER_E'],
    'sector': ['제조업', '제조업', '은행', '제조업', '제조업']
})

# 4. Stage 5 스크리너 실행
try:
    screener = FinancialHealthScreener(params=mock_params)
    mock_loader = MockLoader()
    base_date = date(2026, 7, 31)
    
    passed_df = screener.run(mock_input_df, mock_loader, base_date)
    
    print("📊 [Stage 5 입력 데이터]")
    display(mock_input_df)
    
    print("\n✅ [Stage 5 재무 건전성 필터링 결과]")
    display(passed_df)

except Exception as e:
    print(f"❌ 에러 발생: {e}")